# 01. Automated Program Repair & SWE-bench Lite with Darwin-Evolab

This notebook demonstrates Darwin-Evolab's **Software Automated Program Repair (APR)** pipeline:
1. **Spectrum-Based Fault Localization (SBFL)** using the Ochiai metric
2. **Domain Adapters** for programmatic problem specification
3. **SWE-bench Lite** ingestion and evolutionary patch synthesis producing unified Git diffs


In [1]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

import evolab
print(f'Loaded Darwin-Evolab version: {evolab.__version__}')


Loaded Darwin-Evolab version: 0.5.0


## 1. Spectrum-Based Fault Localization (SBFL Ochiai)

Before mutating code, Darwin-Evolab computes suspicion weights per statement using coverage spectra from passing and failing tests:
\text{Ochiai}(s) = \frac{\text{failed}(s)}{\sqrt{\text{total\_failed} \times (\text{failed}(s) + \text{passed}(s))}}


In [2]:
from evolab.suspicion import compute_ochiai_score, build_suspicion_map

# Sample test spectrum: line 3 is executed only by the failing test
test_spectra = [
    ({1, 2, 4}, True),   # Test 1 passed
    ({1, 2, 4}, True),   # Test 2 passed
    ({1, 2, 3, 4}, False), # Test 3 failed
]

sample_code = '''def divide(a, b):
    if b == 0:
        return None
    return a / b
'''
smap = build_suspicion_map(sample_code, test_spectra)

print('Ochiai Suspicion Scores per line:')
for line, score in sorted(smap.line_scores.items()):
    print(f'  Line {line:2d}: Ochiai = {score:.4f}')


Ochiai Suspicion Scores per line:
  Line  1: Ochiai = 0.5774
  Line  2: Ochiai = 0.5774
  Line  3: Ochiai = 1.0000
  Line  4: Ochiai = 0.5774


## 2. Ingesting Problems via Domain Adapters

Darwin-Evolab decouples problem ingestion from evolutionary algorithms via domain adapters.


In [3]:
from evolab.adapters import get_domain_adapter

adapter = get_domain_adapter('software_repair')
spec = adapter.parse_spec('click_cli_parser')

print(f'Domain:          {adapter.name}')
print(f'Target File:     {spec.target_file}')
print(f'Target Function: {spec.func_name}')
print(f'Total Tests:     {len(spec.tests)}')


Domain:          software_repair
Target File:     cli_parser.py
Target Function: parse_cli
Total Tests:     4


## 3. Ingesting Real SWE-bench Lite Benchmarks

Darwin-Evolab includes offline pre-registered test fixtures for standard SWE-bench Lite issues.


In [4]:
from evolab.swe_bench import SWEBenchAdapter

swe_adapter = SWEBenchAdapter()
fixture_path = repo_root / 'src' / 'evolab' / 'fixtures' / 'swe_bench' / 'sympy__sympy_13480.json'
instance = swe_adapter.parse_spec(str(fixture_path))

print(f'Instance ID: {instance.instance_id}')
print(f'Repository:  {instance.repo}')
print(f'Problem:     {instance.problem_statement[:85]}...')


Instance ID: sympy__sympy-13480
Repository:  sympy/sympy
Problem:     coth(log(tan(x))) evaluation error when x is on the boundary. The substitution condit...


## 4. Running Targeted Evolutionary Repair & Generating Git Patches

We now execute the repair engine. The search evaluates candidate AST mutations against the fail-to-pass test suite and holdout regression tests.


In [5]:
resolution = swe_adapter.solve_instance(instance, max_evals=16)

print(f'Resolved:            {resolution.resolved}')
print(f'FAIL_TO_PASS Passed: {resolution.fail_to_pass_passed}')
print(f'PASS_TO_PASS Clean:  {resolution.pass_to_pass_clean}')
print(f'Evaluations Used:    {resolution.evaluations_used}')
print(f'Execution Time:      {resolution.execution_time_seconds:.4f}s')
print('\n--- Generated Git Patch ---')
print(resolution.generated_patch)


Resolved:            True
FAIL_TO_PASS Passed: True
PASS_TO_PASS Clean:  True
Evaluations Used:    3
Execution Time:      0.0050s

--- Generated Git Patch ---
From: darwin-evolab <repair@evolab.local>
Date: Thu, 10 Sep 2026 17:06:21 +0000
Subject: [PATCH] fix: automated repair by darwin-evolab

---
 1 file(s) changed

--- a/sympy/functions/elementary/hyperbolic.py
+++ b/sympy/functions/elementary/hyperbolic.py
@@ -1,5 +1,4 @@
 def eval_coth_arg(val: float, is_real: bool) -> float:
-    # Boundary check for zero arguments in symbolic hyperbolic evaluation
-    if val > 0.0:  # Bug: should be val >= 0.0 to include boundary
+    if val >= 0.0:
         return 1.0
-    return 0.0
+    return 0.0
